# 🧪 المختبر 03 — PK/PD للمضادات الحيوية

## من MIC إلى الجرعة المثلى

**الفصل التاسع** — علم الأدوية الكيميائي

### 🎯 السؤال الجوهري
لماذا نعطي الجنتامايسين مرة يوميًا رغم أن عمر نصفه 2-3 ساعات؟
**الجواب:** مؤشر الفعالية يختلف (Cmax/MIC vs fT>MIC).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
print("✅ جاهز")

## 📚 المؤشرات الثلاثة

| المؤشر | الفئة | الهدف |
|---|---|---|
| Cmax/MIC | أمينوغليكوزيد | ≥ 10 |
| fT>MIC | β-لاكتام | ≥ 50% |
| AUC/MIC | فانكومايسين | ≥ 400 |

In [ ]:
def pk_iv(dose, vd, cl, t_max=24, n=2000):
    k = cl / vd
    t = np.linspace(0, t_max, n)
    c = (dose / vd) * np.exp(-k * t)
    return t, c

def pk_infusion(dose, vd, cl, t_inf=0.5, t_max=24, n=2000):
    k = cl / vd
    t = np.linspace(0, t_max, n)
    c = np.zeros_like(t)
    mask = t <= t_inf
    c[mask] = (dose / (vd * k * t_inf)) * (1 - np.exp(-k * t[mask]))
    c_end = (dose / (vd * k * t_inf)) * (1 - np.exp(-k * t_inf))
    c[~mask] = c_end * np.exp(-k * (t[~mask] - t_inf))
    return t, c

def metrics(t, c, mic, interval=24):
    cmax = float(c.max())
    auc = float(np.trapz(c, t))
    above = c > mic
    if above.any():
        transitions = np.diff(above.astype(int))
        starts = np.where(transitions == 1)[0] + 1
        ends = np.where(transitions == -1)[0] + 1
        if above[0]:
            starts = np.concatenate([[0], starts])
        if above[-1]:
            ends = np.concatenate([ends, [len(above)]])
        ft_h = float(sum(t[e-1] - t[s] for s, e in zip(starts, ends)))
    else:
        ft_h = 0.0
    return {
        'Cmax': round(cmax, 2),
        'AUC': round(auc, 1),
        'fT>MIC (%)': round(ft_h / interval * 100, 1),
        'Cmax/MIC': round(cmax / mic, 1),
        'AUC/MIC': round(auc / mic, 0),
    }

In [ ]:
# Gentamicin
t_g, c_g = pk_infusion(350, 17.5, 5.25, t_inf=0.5)
m_g = metrics(t_g, c_g, mic=2.0)
print("🔷 Gentamicin 350 mg:")
for k, v in m_g.items():
    print(f"  {k}: {v}")
print(f"  الحكم: {'✅ فعال' if m_g['Cmax/MIC'] >= 10 else '❌ غير فعال'}")

In [ ]:
# Ceftriaxone
t_c, c_c = pk_infusion(2000, 10, 1.0, t_inf=0.5)
m_c = metrics(t_c, c_c, mic=1.0)
print("🔶 Ceftriaxone 2 g:")
for k, v in m_c.items():
    print(f"  {k}: {v}")
print(f"  الحكم: {'✅ فعال' if m_c['fT>MIC (%)'] >= 50 else '❌ غير فعال'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.plot(t_g, c_g, 'b-', linewidth=2.5)
ax.axhline(2.0, color='red', linestyle='--', label='MIC = 2.0')
ax.fill_between(t_g, 2.0, c_g, where=(c_g > 2.0), alpha=0.3, color='green')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Conc (mg/L)')
ax.set_title(f'Gentamicin — Cmax/MIC = {m_g["Cmax/MIC"]}')
ax.legend()

ax = axes[1]
ax.plot(t_c, c_c, 'orange', linewidth=2.5)
ax.axhline(1.0, color='red', linestyle='--', label='MIC = 1.0')
ax.fill_between(t_c, 1.0, c_c, where=(c_c > 1.0), alpha=0.3, color='green')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Conc (mg/L)')
ax.set_title(f'Ceftriaxone — fT>MIC = {m_c["fT>MIC (%)"]}%')
ax.legend()

plt.tight_layout()
plt.show()

## 🎯 الدرس

كلاهما فعال — لكن لأسباب مختلفة:
- **Gentamicin:** Cmax/MIC عالٍ (قتل سريع) — مرة يوميًا
- **Ceftriaxone:** fT>MIC طويل (تأثير مستدام) — مرة يوميًا

**المؤشر الحاكم يحدد الجدول الزمني، لا عمر النصف.**